# Week 4: Machine Learning Model Development and Evaluation
**Prepared by:** Tushar Sharma

Task: predict `Total Spent` using transaction attributes. This is a regression task. Because total spend is closely related to price × quantity, interpret high scores carefully.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv(Path("../data/retail_store_sales_raw.csv"))
for col in ["Price Per Unit", "Quantity", "Total Spent"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
display(df.head())
print("Shape:", df.shape)
print("Missing values:")
display(df.isna().sum())

## 1. Select target and features
Target: `Total Spent`. Predictors: unit price, quantity, category, payment method, location, and discount status. Rows without the target are excluded; missing predictors are handled inside a preprocessing pipeline.

In [ ]:
target = "Total Spent"
features = ["Price Per Unit", "Quantity", "Category", "Payment Method", "Location", "Discount Applied"]
model_df = df[features + [target]].dropna(subset=[target]).copy()
X, y = model_df[features], model_df[target]
numeric_features = ["Price Per Unit", "Quantity"]
categorical_features = ["Category", "Payment Method", "Location", "Discount Applied"]

preprocess = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features)
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))

## 2. Baseline model and tuned Random Forest
Use the same train/test split for fair comparison. GridSearchCV selects hyperparameters using 5-fold cross-validation on training data only.

In [ ]:
baseline = Pipeline([("preprocess", preprocess), ("model", LinearRegression())])
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)

rf = Pipeline([("preprocess", preprocess),
               ("model", RandomForestRegressor(random_state=42, n_jobs=-1))])
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10],
    "model__min_samples_leaf": [1, 3]
}
cv = KFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(rf, param_grid, cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1)
search.fit(X_train, y_train)
best_model = search.best_estimator_
pred = best_model.predict(X_test)
print("Best parameters:", search.best_params_)

In [ ]:
def metric_row(name, actual, predicted):
    return {
        "Model": name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted)
    }

results = pd.DataFrame([
    metric_row("Linear Regression baseline", y_test, base_pred),
    metric_row("Tuned Random Forest", y_test, pred)
])
display(results.round(3))

## Interpretation and limitations
MAE and RMSE measure average prediction error (lower is better); R² measures the share of variation explained on the held-out test set (higher is generally better). Select a model based on test performance and practical simplicity. Since `Total Spent` is commonly determined by `Price Per Unit × Quantity`, performance can be inflated by this built-in relationship. Do not claim general business forecasting ability without testing on external or future data.